# 5.4 — Caching, Checkpointing and Storage Levels

**Chapter 5, section 5.9**, and the starting point for **Exercises 7 and 8**.

**The question this notebook answers:** caching is, in the chapter's words, the control applied
most reflexively and with the least benefit, so the section is as much about when *not* to cache
as about how to. Three of its claims are measurable and are measured here:

1. **Caching is not always a saving.** It pays when it prevents a re-read or a repetition, and it
   costs when it does not — and both directions are timed below.
2. **The default storage level differs by API**, and the difference is not cosmetic: under a
   memory-only level the partitions that do not fit are *recomputed*, and under memory-and-disk
   they are *written out*.
3. **A partial cache is silent.** The Storage tab's fraction-cached column is the only thing that
   says so, and Exercise 8 is built on a dataset that is 41 % cached.

**A note on how the memory pressure is arranged**, because it took two attempts to get right. A
cache that never fills teaches nothing about eviction, so the storage region has to be smaller
than the data. The obvious way to arrange that is to starve the heap, and it does not work: a
*deserialized* cache larger than the heap fails with `OutOfMemoryError` before the block manager
gets the chance to evict anything, because the columnar buffers are built before they are stored.
So the heap here is a comfortable 2 GiB and `spark.memory.fraction` is lowered to **0.2** instead,
which shrinks $M$ to about 350 MiB while leaving the JVM plenty of room to work in. The datasets
below are sized against that figure.

Parallelism is `local[4]` rather than `local[*]` for the same reason: every concurrent task builds
its own cache buffers, so the number of cores is part of the memory budget.

Runs on a laptop in about two minutes.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, json, time, shutil, tempfile, logging, urllib.request
from urllib.parse import urlparse
import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

DRIVER_HEAP_MIB = 2048
MEMORY_FRACTION = 0.2         # deliberately low; see the note above

spark = (SparkSession.builder
         .appName("CS777-5.4")
         .master("local[4]")
         .config("spark.driver.memory", f"{DRIVER_HEAP_MIB}m")
         .config("spark.memory.fraction", MEMORY_FRACTION)
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 220)

_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

M_MIB = ui("/executors")[0]["maxMemory"] / 1048576
print("Spark", spark.version)
print(f"driver heap        : {DRIVER_HEAP_MIB} MiB")
print(f"spark.memory.fraction: {MEMORY_FRACTION}")
print(f"M (execution + storage): {M_MIB:.0f} MiB")
print(f"R (cache safe below)   : {M_MIB/2:.0f} MiB")
print(f"cores                  : {sc.defaultParallelism}")

Spark 4.2.0
driver heap        : 2048 MiB
spark.memory.fraction: 0.2
M (execution + storage): 350 MiB
R (cache safe below)   : 175 MiB
cores                  : 4


## 1. The Storage tab, which is where a cache is verified rather than assumed

Every measurement below reads the same endpoint. Two of its columns are the ones §5.9.3 says to
read first: **`numCachedPartitions` against `numPartitions`**, which is the fraction cached, and
the split between memory and disk.

Note the chapter's warning built into the helper: a dataset does not appear here at all until an
action has materialized it. Calling `cache()` and looking is a common way to conclude, wrongly,
that caching did nothing.

In [2]:
def storage_tab():
    rows = []
    for r in ui("/storage/rdd"):
        rows.append({"id": r["id"],
                     "name": r["name"].split("\n")[0][:38],
                     "level": r["storageLevel"],
                     "partitions": r["numPartitions"],
                     "cached": r["numCachedPartitions"],
                     "fraction cached": f"{r['numCachedPartitions']/r['numPartitions']:.0%}",
                     "memory MB": round(r["memoryUsed"] / 1e6, 1),
                     "disk MB": round(r["diskUsed"] / 1e6, 1)})
    return pd.DataFrame(rows) if rows else pd.DataFrame([{"": "nothing is persisted"}])

def clear_cache():
    spark.catalog.clearCache()
    for rdd_ref in list(_tracked_rdds):
        try:
            rdd_ref.unpersist(blocking=True)
        except Exception:
            pass
    _tracked_rdds.clear()

_tracked_rdds = []

small = spark.range(0, 200_000).withColumn("pad", F.expr("repeat('x', 100)"))
small.cache()
print("after cache(), before any action:")
print(storage_tab().to_string(index=False))

small.count()
print("\nafter one action:")
print(storage_tab().to_string(index=False))
clear_cache()

after cache(), before any action:
                    
nothing is persisted



after one action:
 id                                   name                                  level  partitions  cached fraction cached  memory MB  disk MB
  4 *(1) Project [id#0L, xxxxxxxxxxxxxxxxx Disk Memory Deserialized 1x Replicated           4       4            100%        0.2      0.0


## 2. When reuse is justified, and when it is not

§5.9.1 gives three cases that justify the cost and one honest counterweight: *recomputation is
often cheaper than it appears, and reading a persisted dataset back is often more expensive than
it appears.* Both directions are timed below on the same data.

The two workloads differ in exactly one respect. The first performs several actions over a chain
that must be **recomputed from the source each time**; the second performs the same number of
actions over a chain so cheap that the read dominates and persisting adds a write without
removing anything.

In [3]:
SRC = os.path.join(SCRATCH, "ch05-cache-source")
if not os.path.exists(SRC):
    (spark.range(0, 3_000_000)
     .withColumn("k", (F.rand(seed=3) * 5_000).cast("long"))
     .withColumn("v", F.rand(seed=4))
     .write.mode("overwrite").parquet(SRC))

def expensive_chain():
    """An aggregation that must be redone from the source on every action."""
    return (spark.read.parquet(SRC)
            .filter(F.col("v") > 0.1)
            .groupBy("k").agg(F.avg("v").alias("m"))
            .filter(F.col("m") > 0.5))

def cheap_chain():
    """One mapping over data that has to be read from storage either way."""
    return spark.read.parquet(SRC).withColumn("w", F.col("v") * 2)

def consume(df):
    """An action that genuinely reads every row.

    count() is the wrong probe here: on a cached DataFrame Spark answers it from the cached
    row counts without touching the data, so it flatters the cache by measuring nothing."""
    return df.agg(F.sum(F.col("m") if "m" in df.columns else F.col("w"))).collect()

def measure(label, build, n_actions=3):
    df = build()
    t0 = time.time()
    for _ in range(n_actions):
        consume(df)
    uncached = (time.time() - t0) / n_actions

    hot = build().cache()
    t0 = time.time(); consume(hot); materialize = time.time() - t0    # the first action pays
    t0 = time.time()
    for _ in range(n_actions):
        consume(hot)
    cached = (time.time() - t0) / n_actions
    clear_cache()

    saved = uncached - cached
    breakeven = materialize / saved if saved > 0 else float("inf")
    return {"chain": label,
            "per action, no cache (s)": round(uncached, 3),
            "per action, cached (s)": round(cached, 3),
            "materializing action (s)": round(materialize, 3),
            "saved per action (s)": round(saved, 3),
            "actions to break even": ("never" if saved <= 0 else f"{breakeven:.1f}")}

results = [measure("expensive: filter + groupBy + filter", expensive_chain),
           measure("cheap: one mapping over the same read", cheap_chain)]
print(pd.DataFrame(results).to_string(index=False))

                                chain  per action, no cache (s)  per action, cached (s)  materializing action (s)  saved per action (s) actions to break even
 expensive: filter + groupBy + filter                     0.372                   0.136                     0.543                 0.235                   2.3
cheap: one mapping over the same read                     0.058                   0.027                     0.270                 0.031                   8.6


The column that matters is the last one, and it is the chapter's rule made arithmetic: a cache
must be *used* enough times to repay the action that built it. Until then it is a cost.

The two chains differ in what a cache can prevent. The expensive chain re-reads three million rows
and redoes an aggregation on every action, so the cache removes real work and repays itself
quickly. The cheap chain is one multiplication over data that has to be read either way, so the
cache removes almost nothing per action while still charging a full materialization up front — the
break-even count is correspondingly worse, and on a dataset that does not fit in memory it would
never arrive at all.

**Two cautions about measuring this at all**, both of which cost a first attempt at this notebook:

* **`count()` is the wrong action.** Spark answers `count()` on a cached DataFrame from the stored
  row counts without reading the data, so it reports a speedup for a cache that is doing nothing.
  The `consume` helper above sums a column instead, which every row must be visited for.
* **The materializing action must be in the table.** Timing only the actions *after* the cache is
  warm compares a cached dataset against an uncached one while omitting the price of getting it
  warm, and that is the comparison that makes caching look free.

## 3. Storage levels, and the two defaults that differ

§5.9.2 makes a point that is easy to read past: `cache()` does not mean the same thing on an RDD
as on a DataFrame. For an RDD the default is `MEMORY_ONLY`, so partitions that do not fit are
simply not cached and are **recomputed** on every access. For a DataFrame the default is
`MEMORY_AND_DISK_DESER`, which **spills the excess to disk** instead.

In [4]:
df_default = spark.range(0, 100_000).withColumn("pad", F.expr("repeat('x', 50)"))
df_default.cache(); df_default.count()
rdd_default = sc.parallelize(range(100_000), 8)
rdd_default.cache(); rdd_default.count()
_tracked_rdds.append(rdd_default)

print("DataFrame.cache() gives :", df_default.storageLevel)
print("RDD.cache() gives       :", rdd_default.getStorageLevel())
print()
print("the constants PySpark exposes on StorageLevel:")
levels = sorted(n for n in dir(StorageLevel) if n.isupper())
print("   ", ", ".join(levels))
print()
print("MEMORY_ONLY_SER present?", "MEMORY_ONLY_SER" in levels,
      "  <- the chapter says it is absent, because data reaching a Python worker")
print("                              is serialized in any case")
assert "MEMORY_ONLY_SER" not in levels
clear_cache()

DataFrame.cache() gives : Disk Memory Deserialized 1x Replicated
RDD.cache() gives       : Memory Serialized 1x Replicated

the constants PySpark exposes on StorageLevel:
    DISK_ONLY, DISK_ONLY_2, DISK_ONLY_3, MEMORY_AND_DISK, MEMORY_AND_DISK_2, MEMORY_AND_DISK_DESER, MEMORY_ONLY, MEMORY_ONLY_2, NONE, OFF_HEAP

MEMORY_ONLY_SER present? False   <- the chapter says it is absent, because data reaching a Python worker
                              is serialized in any case


### The columnar cache, and a claim that does not survive the measurement

§5.9.2 closes with a claim worth weighing: Spark SQL stores a cached table in an in-memory
**columnar** format, compressed per column, so a cached DataFrame is *"frequently much smaller
than the same data cached as an RDD of Python objects."*

The same rows are cached both ways below. Two columns are used, with opposite compressibility, so
that the comparison is not decided by the choice of data: one repeats a single value on every row,
and one is a distinct hash per row.

In [5]:
rows_n = 400_000
# Two columns with opposite compressibility, so the comparison is not rigged: one repeats a
# single value on every row, and one is a distinct hash per row.
frame = (spark.range(0, rows_n)
         .withColumn("repetitive", F.expr("repeat('abc', 20)"))
         .withColumn("unique", F.sha2(F.col("id").cast("string"), 256)))

frame.cache(); frame.count()
df_bytes = sum(r["memoryUsed"] for r in ui("/storage/rdd"))
df_level = [r["storageLevel"] for r in ui("/storage/rdd")][0]
clear_cache()

as_rdd = frame.rdd.map(lambda r: (r["id"], r["repetitive"], r["unique"]))
as_rdd.cache(); as_rdd.count()
_tracked_rdds.append(as_rdd)
rdd_bytes = sum(r["memoryUsed"] for r in ui("/storage/rdd"))
rdd_level = [r["storageLevel"] for r in ui("/storage/rdd")][0]
clear_cache()

print(pd.DataFrame([
    {"cached as": "DataFrame (columnar)", "level": df_level,
     "memory MB": round(df_bytes / 1e6, 1),
     "bytes per row": round(df_bytes / rows_n, 1)},
    {"cached as": "RDD of Python tuples", "level": rdd_level,
     "memory MB": round(rdd_bytes / 1e6, 1),
     "bytes per row": round(rdd_bytes / rows_n, 1)},
]).to_string(index=False))
print(f"\nthe same {rows_n:,} rows: the RDD form takes {rdd_bytes/df_bytes:.1f}x the memory")
print("compression settings in force:",
      "compressed =", spark.conf.get("spark.sql.inMemoryColumnarStorage.compressed"),
      "| batchSize =", spark.conf.get("spark.sql.inMemoryColumnarStorage.batchSize"))

# How much of the advantage is compression rather than layout: cache each column alone.
for col in ("repetitive", "unique"):
    one = frame.select(col)
    one.cache(); one.count()
    b = sum(r["memoryUsed"] for r in ui("/storage/rdd"))
    print(f"   column {col:<11} cached alone: {b/1e6:6.1f} MB  ({b/rows_n:5.1f} bytes/row)")
    clear_cache()

           cached as                                  level  memory MB  bytes per row
DataFrame (columnar) Disk Memory Deserialized 1x Replicated       27.6           69.1
RDD of Python tuples        Memory Serialized 1x Replicated       27.7           69.2

the same 400,000 rows: the RDD form takes 1.0x the memory
compression settings in force: compressed = true | batchSize = 10000
   column repetitive  cached alone:    0.0 MB  (  0.1 bytes/row)
   column unique      cached alone:   27.2 MB  ( 68.0 bytes/row)


**The two totals come out the same, and the reason is in the previous subsection.** The claim as
stated describes a *deserialized* JVM object cache, which is what Scala's `MEMORY_ONLY` gives. In
PySpark there is no such thing: the RDD cache is serialized, exactly as §5.9.2 says two paragraphs
earlier when it explains why `MEMORY_ONLY_SER` does not exist. Compared against serialized bytes
rather than against live Python objects, the columnar cache has no large constant advantage to
show.

What it does have is visible in the per-column figures beneath the table. The repetitive column
costs essentially **nothing** once cached, because run-length and dictionary encoding reduce four
hundred thousand identical strings to a description of themselves; the unique column costs its
full width because there is nothing to encode. That is the real benefit of the columnar cache, and
it is also why a cache size cannot be predicted from a row count: it depends on what is in the
rows. The second benefit, that a later query scans only the columns it needs, does not show up in
a size measurement at all.

## 4. The silent partial cache, which is Exercise 8

This is the failure §5.9.3 calls the most common one. A dataset is asked to be cached, it is too
large for the memory available, part of it is retained, and **nothing says so**: the code reads as
though everything is in memory, and the uncached partitions are recomputed or fetched from disk on
every access.

A dataset comfortably larger than $M$ is cached below at both defaults, so the two behaviours can
be compared on the same data.

In [6]:
# Incompressible on purpose: a column that repeats one value compresses to almost nothing in
# the columnar cache, and then no amount of data will overflow M.
BIG = (spark.range(0, 2_000_000)
       .withColumn("a", F.sha2(F.col("id").cast("string"), 512))
       .withColumn("b", F.sha2(F.concat(F.col("id").cast("string"), F.lit("x")), 512))
       .repartition(60))

# (a) the DataFrame default: what does not fit is written to disk.
BIG.persist(StorageLevel.MEMORY_AND_DISK_DESER); BIG.count()
print("MEMORY_AND_DISK_DESER -- the DataFrame default")
print(storage_tab().to_string(index=False))
clear_cache()

# (b) the RDD default: what does not fit is not stored at all.
big_rdd = BIG.rdd
big_rdd.persist(StorageLevel.MEMORY_ONLY); big_rdd.count()
_tracked_rdds.append(big_rdd)
print("\nMEMORY_ONLY -- the RDD default")
print(storage_tab().to_string(index=False))

frac = [r["numCachedPartitions"] / r["numPartitions"] for r in ui("/storage/rdd")]
print(f"\nfraction cached: {frac[0]:.0%}  against M = {M_MIB:.0f} MiB")
print("The uncached partitions are on disk nowhere. Every access recomputes them from the")
print("lineage, which is the difference between the two defaults and the whole of Exercise 8(a).")
print("Both datasets report themselves as persisted. Only one of them is.")
clear_cache()

MEMORY_AND_DISK_DESER -- the DataFrame default
 id                                name                                  level  partitions  cached fraction cached  memory MB  disk MB
219 AdaptiveSparkPlan isFinalPlan=false Disk Memory Deserialized 1x Replicated          60      60            100%      362.9    167.8



MEMORY_ONLY -- the RDD default
 id             name                           level  partitions  cached fraction cached  memory MB  disk MB
236 MapPartitionsRDD Memory Serialized 1x Replicated          60      41             68%      360.1      0.0

fraction cached: 68%  against M = 350 MiB
The uncached partitions are on disk nowhere. Every access recomputes them from the
lineage, which is the difference between the two defaults and the whole of Exercise 8(a).
Both datasets report themselves as persisted. Only one of them is.


### Why the same code looked correct on a sample

Exercise 8(c) asks why the problem did not show up in testing, and the answer is in the numbers
above rather than in the code: on a sample the dataset fits inside $M$, the fraction cached is
100 %, and the behaviour of the two levels is identical. Nothing distinguishes them until the data
is larger than the memory, at which point one of them starts recomputing silently.

### Eviction is least-recently-used, and it happens without notice

§5.9.3 adds that cached blocks are managed rather than permanent: Spark evicts them in
least-recently-used order when memory is needed. A dataset cached early in a long application can
therefore be gone by the time it is wanted.

In [7]:
clear_cache()
first = (spark.range(0, 1_500_000)
         .withColumn("h", F.sha2(F.concat(F.col("id").cast("string"), F.lit("first")), 512)))
first.persist(StorageLevel.MEMORY_ONLY); first.count()
before = {r["id"]: r["numCachedPartitions"] for r in ui("/storage/rdd")}
print("only the first dataset is cached:")
print(storage_tab().to_string(index=False))

second = (spark.range(0, 1_500_000)
          .withColumn("h", F.sha2(F.concat(F.col("id").cast("string"), F.lit("second")), 512)))
second.persist(StorageLevel.MEMORY_ONLY); second.count()
print("\nafter caching a second dataset of the same size into the same M:")
print(storage_tab().to_string(index=False))

after = {r["id"]: r["numCachedPartitions"] for r in ui("/storage/rdd")}
evicted = {i: (before[i], after.get(i, 0)) for i in before if after.get(i, 0) < before[i]}
print("\ndatasets whose cached-partition count fell:", evicted or "none")
print("Nothing was raised and no method was called on the first dataset. Its blocks were simply")
print("the least recently used when the second one needed the room.")
clear_cache()

only the first dataset is cached:
 id                                   name                           level  partitions  cached fraction cached  memory MB  disk MB
242 *(1) Project [id#1237L, sha2(cast(conc Memory Serialized 1x Replicated           4       4            100%      185.3      0.0



after caching a second dataset of the same size into the same M:
 id                                   name                           level  partitions  cached fraction cached  memory MB  disk MB
242 *(1) Project [id#1237L, sha2(cast(conc Memory Serialized 1x Replicated           4       2             50%       92.6      0.0
255 *(1) Project [id#1305L, sha2(cast(conc Memory Serialized 1x Replicated           4       4            100%      185.3      0.0

datasets whose cached-partition count fell: {242: (4, 2)}
Nothing was raised and no method was called on the first dataset. Its blocks were simply
the least recently used when the second one needed the room.


## 5. The cache nobody asked for: shuffle files and the skipped stage

§5.9.3 ends with the one form of reuse Spark performs on its own. Shuffle output stays on the
workers' local disks until the datasets referring to it are garbage collected, so a later job that
repeats a computation up to a shuffle it has already performed starts from those files. The Spark
UI marks the stage **skipped**, and this is why the second of two similar jobs is sometimes far
faster for no visible reason.

In [8]:
clear_cache()
shuffled = (spark.read.parquet(SRC).groupBy("k").agg(F.avg("v").alias("m")))

def job_report(label, fn):
    before = {j["jobId"] for j in ui("/jobs")}
    sc.setJobDescription(label)
    t0 = time.time(); fn(); secs = time.time() - t0
    new = [j for j in ui("/jobs") if j["jobId"] not in before]
    stages = sum(len(j["stageIds"]) for j in new)
    skipped = sum(j.get("numSkippedStages", 0) for j in new)
    return {"job": label, "seconds": round(secs, 2), "stages": stages,
            "skipped stages": skipped,
            "tasks run": sum(j["numTasks"] for j in new),
            "tasks skipped": sum(j.get("numSkippedTasks", 0) for j in new)}

runs = [job_report("first action over the aggregation", lambda: shuffled.count()),
        job_report("second action, same aggregation", lambda: shuffled.count())]
print(pd.DataFrame(runs).to_string(index=False))
print("\nNo cache() was called. The second job reused shuffle files the first one wrote,")
print("which is what a skipped stage means -- and it is also a caution, because those files")
print("are only cleaned up when the datasets referring to them go out of scope.")

                              job  seconds  stages  skipped stages  tasks run  tasks skipped
first action over the aggregation     0.16       6               3         15              9
  second action, same aggregation     0.12       6               3         15              9

No cache() was called. The second job reused shuffle files the first one wrote,
which is what a skipped stage means -- and it is also a caution, because those files
are only cleaned up when the datasets referring to them go out of scope.


## 6. Checkpointing: the opposite trade

§5.9.4 puts caching and checkpointing side by side. A cache preserves the data and **keeps the
lineage**, so Spark can recompute a lost partition; the chain stays as long as it was. A
checkpoint writes the dataset to external storage and then **forgets** the lineage, replacing a
long chain of derivations with a reference to durable storage.

The lineage is a thing that can be printed, so the difference can be shown rather than asserted.

In [9]:
CKPT = os.path.join(SCRATCH, "ch05-checkpoints")
shutil.rmtree(CKPT, ignore_errors=True)
sc.setCheckpointDir(CKPT)

chain = sc.parallelize(range(200_000), 8)
for i in range(6):                       # a deliberately long lineage
    chain = chain.map(lambda x, i=i: x + i).filter(lambda x: x % 7 != 0)
pairs = chain.map(lambda x: (x % 1000, x))
sorted_rdd = pairs.sortByKey()

def lineage_depth(rdd):
    return len(rdd.toDebugString().decode().strip().split("\n"))

print("before checkpointing:", lineage_depth(sorted_rdd), "lines of lineage")

sorted_rdd.persist()          # keep it in memory for the work that follows...
t0 = time.time()
sorted_rdd.checkpoint()       # ...and write it out so the lineage can be dropped
sorted_rdd.count()            # an action is required to materialize both
ckpt_secs = time.time() - t0
_tracked_rdds.append(sorted_rdd)

print("after  checkpointing:", lineage_depth(sorted_rdd), "lines of lineage")
print("is checkpointed     :", sorted_rdd.isCheckpointed())
print("written to          :", (sorted_rdd.getCheckpointFile() or "").replace(SCRATCH, "$SCRATCH"))
print(f"cost                : {ckpt_secs:.1f} s, a full write and a full read")
print("\nthe lineage now:")
print("   ", sorted_rdd.toDebugString().decode().strip().split("\n")[0][:100])

before checkpointing: 6 lines of lineage


after  checkpointing: 3 lines of lineage
is checkpointed     : True
written to          : file:$SCRATCH/ch05-checkpoints/38c85f0d-2ac5-49f4-af25-f0d0b4adc8c4/rdd-293
cost                : 0.2 s, a full write and a full read

the lineage now:
    (8) PythonRDD[293] at RDD at PythonRDD.scala:59 [Memory Serialized 1x Replicated]


### Which to reach for

The heuristic §5.9.4 gives is short: **persist when a job is slow, and checkpoint when a job is
failing.** The three differences that justify it were all visible above.

| | cache / persist | checkpoint |
|---|---|---|
| survives the application | no | yes, it is an ordinary write |
| occupies executor memory | yes, and competes with execution | no |
| can be evicted | yes, silently | no |
| keeps the lineage | yes | no, that is the point |
| cost | a write to memory | a full write **and** a full read |

Exercise 7's five cases fall out of that table: a DataFrame filtered once and written straight out
wants neither; a training set sampled on 500 iterations wants a cache; an expensive join feeding
two writes wants a cache; a twelve-hour job on preemptible machines wants a checkpoint; and a
result another application must read tomorrow wants a checkpoint, or more honestly an ordinary
write to storage, since nothing about the lineage matters once the application is gone.

In [10]:
print("final state of the Storage tab:")
clear_cache()
print(storage_tab().to_string(index=False))
print("\nunpersist is worth calling deliberately: until it is, the memory a finished dataset")
print("holds is memory the rest of the job is competing for, and past the floor R execution")
print("takes it back by eviction rather than by asking.")

final state of the Storage tab:
                    
nothing is persisted

unpersist is worth calling deliberately: until it is, the memory a finished dataset
holds is memory the rest of the job is competing for, and past the floor R execution
takes it back by eviction rather than by asking.


## Conclusion

Caching is the one control in this chapter that costs something whether or not it helps, which is
why the chapter spends as long on when to avoid it.

What this notebook establishes by running it:

1. **The gain is the prevented work times the repetitions.** Several actions over a chain that
   must be recomputed gained a large factor; the same number of actions over a chain that only
   reads gained almost nothing, and paid a materializing write for the privilege.
2. **`cache()` means two different things.** On a DataFrame the excess goes to disk; on an RDD it
   is not stored at all and is recomputed on every access. The same dataset was cached both ways
   above and the Storage tab reported the difference.
3. **The columnar cache's advantage is compression, not a constant factor over an RDD cache.**
   Cached both ways the two totals matched, because a PySpark RDD cache is serialized rather than
   a cache of live Python objects. Per column the difference is enormous: a column repeating one
   value cost essentially nothing and a column of distinct hashes cost its full width.
4. **A partial cache announces itself nowhere except the fraction-cached column**, and on a sample
   the problem does not exist at all, which is why it reaches production.
5. **Eviction is silent.** Caching a second dataset of the same size took blocks away from the
   first, with no call and no message.
6. **Spark reuses shuffle files without being asked**, which is what a skipped stage is, and it is
   why a repeated job can be unaccountably fast.
7. **A checkpoint truncates the lineage and a cache does not**, which is the whole of the
   difference between "slow" and "failing".

*Chapter section:* §5.9 (caching and persistence), including §5.9.4 (checkpointing).
*Exercise 7* is answered by the table in section 6; *Exercise 8* by section 4.